# S3, DynamoDB, and SQS — the data + messaging plane, locally

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 33 §33.4–§33.5 · type: walkthrough*

*One-line promise:* exercise the capstone's **storage and queue plane** end-to-end — put/get an object, read/write an item, enqueue/consume a task — against **simulated AWS** with the **real `boto3` call shapes**, and **no AWS account, no spend**.

## 🧠 Why this matters

The async backbone you built by hand in Ch 30–31 — a data layer plus a queue feeding workers — has managed AWS equivalents: **S3** for objects, **DynamoDB** for key–value items, and **SQS** for the API→worker handoff. The catch is that you usually only discover the sharp edges (the two-ARN S3 model, DynamoDB's typed keys, SQS *at-least-once* delivery) by hitting them in production — on a real account, with a real bill. `moto` removes that tax: it runs an in-process fake of AWS that speaks the **same `boto3` API**, so you can rehearse the exact calls the capstone makes — round-tripping an artifact, persisting a session, handing a task to a worker — for free, offline, and deterministically. You learn the call shapes here so the real ones in Ch 36 hold no surprises.

## Objectives & prereqs

**By the end you can:**
- Stand up **S3 + DynamoDB + SQS** in-process with `moto` using the same `boto3` clients you'd point at real AWS.
- Round-trip an artifact through **S3** (upload → list → download → verify bytes).
- Model and query a **DynamoDB** session item by its key.
- Drive the **SQS** send → receive → delete handoff — the API↔worker boundary from Ch 31 — and make the consumer **idempotent** against *at-least-once* delivery.
- Flip the same code at **LocalStack** by setting one `endpoint_url`.

**Prereqs:** [`33-01-iam-and-least-privilege.ipynb`](./33-01-iam-and-least-privilege.ipynb) (the *identity* these calls run under); Ch 30 (data layer), Ch 31 (queues/workers).

**Packages:** `boto3` + `moto` — **not in `requirements.txt`**; install the chapter extras with `pip install boto3 moto`. The notebook *checks* for them in Setup and fails with a friendly message rather than a stack trace.

**Cost:** `local-sim only`. `MOCK=1` (default) runs everything in-process via `moto` — **free, offline, no AWS account.** ⚠️ Pointing these same calls at a *real* AWS account (by unsetting the mock and giving real credentials) **would incur charges** — that's an explicit, opt-in exercise, never the default.

## Setup

In [ ]:
import io
import json
import os
import random
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # reads a git-ignored .env if present; never hardcode keys or credentials

# MOCK=1 (default) runs S3/DynamoDB/SQS entirely in-process via `moto` -> FREE, OFFLINE, no
# AWS account. MOCK=0 would point boto3 at *real* AWS (real credentials, real charges) and is
# intentionally left as an opt-in exercise, not wired up here.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(33)  # determinism for anything we generate

# moto needs *some* credentials present to satisfy boto3's signer, but they are never used
# against real AWS in mock mode. We set obvious throwaways so nothing real can be touched.
REGION = "us-east-1"
os.environ.setdefault("AWS_ACCESS_KEY_ID", "testing")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "testing")
os.environ.setdefault("AWS_DEFAULT_REGION", REGION)

# Same code, real LocalStack: set LOCALSTACK_ENDPOINT (e.g. http://localhost:4566) and the
# clients below pick it up via endpoint_url. Unset -> pure in-process moto.
ENDPOINT_URL = os.getenv("LOCALSTACK_ENDPOINT") or None

BUCKET = "capstone-artifacts"
TABLE = "capstone-sessions"
QUEUE = "capstone-tasks"
DATA = Path("data")

try:
    import boto3
    from moto import mock_aws
    HAVE_AWS_SIM = True
except ImportError:
    HAVE_AWS_SIM = False

if not HAVE_AWS_SIM:
    print("This notebook needs the chapter extras:  pip install boto3 moto")
    print("They are not in requirements.txt by default. Install them, then re-run.")
else:
    print(f"MOCK = {MOCK} | region = {REGION} | endpoint_url = {ENDPOINT_URL or '(in-process moto)'}")
    print(f"bucket={BUCKET}  table={TABLE}  queue={QUEUE}")

### One simulated AWS, three clients

`mock_aws` is the modern `moto` entry point: as a context manager it intercepts **every** `boto3` call inside it and serves it from an in-memory fake of AWS. Crucially, the client code is *identical* to production — `boto3.client("s3")`, `put_object`, `send_message`. That's the whole point: you rehearse the real shapes here, then the only thing that changes for real AWS is that you stop wrapping it in `mock_aws` (and provide real credentials and an eye on the bill).

We open one `mock_aws()` context and keep it active for the rest of the notebook via a small helper, so each step reads like ordinary `boto3`.

In [ ]:
# Enter ONE moto context for the whole notebook so later cells read like plain boto3.
# (In a script you'd use `with mock_aws():` around the work; in a notebook we enter it once.)
_ctx = None


def _client(service):
    """A boto3 client for `service`, pointed at LocalStack if ENDPOINT_URL is set."""
    return boto3.client(service, region_name=REGION, endpoint_url=ENDPOINT_URL)


def _resource(service):
    return boto3.resource(service, region_name=REGION, endpoint_url=ENDPOINT_URL)


if HAVE_AWS_SIM:
    # Only start moto when we're NOT talking to a real LocalStack endpoint.
    if ENDPOINT_URL is None:
        _ctx = mock_aws()
        _ctx.start()
        print("moto: in-process AWS is active (nothing real is reachable).")
    else:
        print(f"Talking to LocalStack at {ENDPOINT_URL} (still no real AWS).")

    s3 = _client("s3")
    ddb = _client("dynamodb")
    sqs = _client("sqs")
    print("clients ready: s3, ddb, sqs")

## 1 · S3 — the artifact round-trip (§33.4)

**S3** is object storage: effectively infinite, cheap, durable — the capstone's home for artifacts, model outputs, and exports. The capstone exports a small JSON artifact; here we create the bucket, upload that artifact, list to confirm it landed, download it back, and verify the bytes round-tripped **unchanged**. These are the exact `boto3` calls (`create_bucket`, `put_object`, `list_objects_v2`, `get_object`) you'd make against real S3.

In [ ]:
if HAVE_AWS_SIM:
    # us-east-1 is special: it must NOT get a LocationConstraint. Every other region must.
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=BUCKET)
    else:
        s3.create_bucket(
            Bucket=BUCKET,
            CreateBucketConfiguration={"LocationConstraint": REGION},
        )

    # The tiny committed fixture from data/ is our artifact.
    artifact_bytes = (DATA / "sample-artifact.json").read_bytes()
    key = "exports/capstone-export-v3.json"

    s3.put_object(Bucket=BUCKET, Key=key, Body=artifact_bytes,
                  ContentType="application/json")

    listed = s3.list_objects_v2(Bucket=BUCKET).get("Contents", [])
    print("objects in bucket:", [o["Key"] for o in listed])

    downloaded = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    print("bytes round-trip identical:", downloaded == artifact_bytes)
    print("artifact version:", json.loads(downloaded)["version"])

**What you just saw.** The bytes you uploaded came back identical — a clean object round-trip through the real S3 API, for free. Note the `us-east-1` quirk in the code: it's the one region that rejects a `LocationConstraint`. `moto` enforces that exactly like AWS, which is precisely why rehearsing here catches the bug before it costs you a failed deploy.

## 2 · DynamoDB — a session item (§33.4)

**DynamoDB** is managed NoSQL key–value: the right home for high-scale, known-access-pattern data like a **session store**. We define a table keyed by `session_id`, then `put_item` a session and read it straight back by key with `get_item`. Two DynamoDB-isms to internalize: attribute values are **typed** (`{"S": ...}` for string, `{"N": ...}` for number, sent as a string), and you query by the **key schema** you declared — not by arbitrary columns.

In [ ]:
if HAVE_AWS_SIM:
    ddb.create_table(
        TableName=TABLE,
        KeySchema=[{"AttributeName": "session_id", "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": "session_id", "AttributeType": "S"}],
        BillingMode="PAY_PER_REQUEST",
    )

    # A session item. Note the typed attribute values: 'S' = string, 'N' = number-as-string.
    ddb.put_item(
        TableName=TABLE,
        Item={
            "session_id": {"S": "sess-001"},
            "user": {"S": "ada"},
            "tokens_in": {"N": "412"},
            "tokens_out": {"N": "88"},
            "status": {"S": "complete"},
        },
    )

    item = ddb.get_item(
        TableName=TABLE,
        Key={"session_id": {"S": "sess-001"}},
    ).get("Item")
    print("read back by key:", item)

**What you just saw.** A single-key put and get, with the typed-attribute envelope DynamoDB requires. That key schema is a *commitment*: DynamoDB is fast and cheap precisely because you designed for the access pattern up front (read a session *by its id*). Try to query it a different way later and you'll need a secondary index — which is the Ch 30 lesson, now in its managed form.

## 3 · SQS — the API↔worker handoff (§33.5)

**SQS** is a simple, reliable managed queue — the natural broker for handing agent tasks from the API to a fleet of workers (the Ch 31 pattern, managed). The lifecycle is three calls: the API `send_message`; a worker `receive_message`; and — only after it has *successfully* done the work — `delete_message`. That last step is the crux: a message you receive but don't delete becomes visible again and gets redelivered. SQS guarantees **at-least-once**, not exactly-once.

In [ ]:
if HAVE_AWS_SIM:
    queue_url = sqs.create_queue(QueueName=QUEUE)["QueueUrl"]

    # API side: enqueue a task for the workers.
    task = {"task_id": "t-7", "session_id": "sess-001", "op": "summarize"}
    sqs.send_message(QueueUrl=queue_url, MessageBody=json.dumps(task))
    print("enqueued:", task)

    # Worker side: receive, do the work, then delete the message.
    resp = sqs.receive_message(QueueUrl=queue_url, MaxNumberOfMessages=1,
                               WaitTimeSeconds=0)
    msg = resp["Messages"][0]
    body = json.loads(msg["Body"])
    print("worker received:", body)

    # ... worker does the work here ...
    sqs.delete_message(QueueUrl=queue_url, ReceiptHandle=msg["ReceiptHandle"])
    print("deleted (acknowledged) task", body["task_id"])

### 🔮 Predict

We just *received and deleted* the only message on the queue. Now we call `receive_message` again, with `WaitTimeSeconds=0` (immediate, **short-poll**).

**Predict:** what does the response contain?
- (a) the same task again,
- (b) a `Messages` key holding an empty list,
- (c) **no** `Messages` key at all (you must use `.get("Messages", [])`).

Write your guess, then run the next cell.

In [ ]:
if HAVE_AWS_SIM:
    empty = sqs.receive_message(QueueUrl=queue_url, WaitTimeSeconds=0)
    print("'Messages' key present?", "Messages" in empty)
    print("safe drain count:", len(empty.get("Messages", [])))

**What you just saw.** The answer is **(c)**: on an empty queue, SQS omits the `Messages` key entirely — it is *not* an empty list. Indexing `resp["Messages"]` blindly is a classic `KeyError` in worker loops; always `resp.get("Messages", [])`. With `WaitTimeSeconds=0` you **short-poll** (return immediately, possibly empty even when messages exist on a distributed queue); set it to 1–20 to **long-poll**, which waits for a message and cuts both empty responses and request cost. Workers should long-poll.

### ⚠️ Pitfall: at-least-once delivery — make the consumer idempotent

Because SQS redelivers (a worker can crash *after* doing the work but *before* `delete_message`, or the visibility timeout can lapse), the **same task can arrive twice**. If processing a task isn't idempotent — say it increments a counter or charges a card — a redelivery double-counts. The fix (Ch 29/31) is to make the *effect* idempotent: record which `task_id`s you've completed and no-op on a repeat. Below we deliberately deliver the same task twice and show a naive consumer over-counting, then an idempotent one getting it right.

In [ ]:
if HAVE_AWS_SIM:
    # Deliver the SAME logical task twice (simulating an at-least-once redelivery).
    dupe = {"task_id": "t-7", "session_id": "sess-001", "op": "summarize"}
    for _ in range(2):
        sqs.send_message(QueueUrl=queue_url, MessageBody=json.dumps(dupe))

    def drain():
        out = []
        while True:
            r = sqs.receive_message(QueueUrl=queue_url, MaxNumberOfMessages=10,
                                    WaitTimeSeconds=0)
            msgs = r.get("Messages", [])
            if not msgs:
                return out
            for m in msgs:
                out.append(m)
                sqs.delete_message(QueueUrl=queue_url, ReceiptHandle=m["ReceiptHandle"])

    messages = drain()

    # Naive consumer: processes every delivery -> over-counts on redelivery.
    naive_count = 0
    for m in messages:
        naive_count += 1

    # Idempotent consumer: skip task_ids already done -> correct under at-least-once.
    done = set()
    idempotent_count = 0
    for m in messages:
        tid = json.loads(m["Body"])["task_id"]
        if tid in done:
            continue  # already processed this logical task; no-op
        done.add(tid)
        idempotent_count += 1

    print(f"deliveries received : {len(messages)}")
    print(f"naive processed     : {naive_count}   (double-counts the redelivery)")
    print(f"idempotent processed: {idempotent_count}   (correct: one logical task)")

**What you just saw.** Two physical deliveries of one logical task: the naive consumer runs the work twice; the idempotent consumer keyed on `task_id` runs it once. *Receiving* a message twice is normal and expected with SQS — the discipline is to make *processing* it twice harmless.

## 🎯 Senior lens: `moto` mirrors API *shape*, not every guarantee

`moto` is the right tool to learn and unit-test against — it reproduces the `boto3` calls, the error shapes, and quirks like the `us-east-1` bucket rule and SQS's missing-`Messages` key. What it does **not** fully reproduce is AWS's *operational* reality: eventual-consistency timing, real visibility-timeout and redrive/DLQ behavior under load, IAM enforcement on every call, throttling, and per-service limits. So treat green-on-`moto` as “my call shapes and logic are right,” not “this is production-ready.” Before you trust it live, run an integration test against **LocalStack** (closer to real behavior) or a throwaway AWS account — which is exactly why this notebook is written so the *same code* points at LocalStack by setting one `endpoint_url`. Mock for fast inner-loop feedback; integration-test for the guarantees that only show up under real conditions.

### Same code, real LocalStack (no AWS account)

Nothing above is `moto`-specific *in the call shapes*. To run this entire notebook against [LocalStack](https://localstack.cloud) instead — a fuller local AWS emulator — you only change the endpoint, not the code:

```bash
# 1. start LocalStack (Docker)
docker run --rm -it -p 4566:4566 localstack/localstack
# 2. point this notebook at it and re-run, top to bottom
export LOCALSTACK_ENDPOINT=http://localhost:4566
```

Our Setup cell reads `LOCALSTACK_ENDPOINT`, skips starting `moto`, and threads `endpoint_url` into every client. Still **no real AWS, no spend** — just a higher-fidelity local target for an integration pass.

## Recap

- `moto`'s `mock_aws` serves the **real `boto3` API** in-process — you rehearse S3/DynamoDB/SQS for free, offline, with no AWS account.
- **S3**: `create_bucket` → `put_object` → `get_object` round-trips bytes unchanged; mind the `us-east-1` no-`LocationConstraint` rule.
- **DynamoDB**: typed attribute values (`{"S"/"N": ...}`) and a key schema you commit to up front; read by that key.
- **SQS**: `send` → `receive` → **`delete`**; an empty queue omits the `Messages` key (use `.get`); long-poll in workers.
- SQS is **at-least-once** — design consumers to be **idempotent** so a redelivery is a no-op.
- `moto` mirrors API *shape*, not every guarantee — integration-test on **LocalStack** or a sandbox account before you trust it in prod. The same code switches targets via one `endpoint_url`.

## Exercises

Use the empty cells below. Each one *changes* something — predict the result first. (Solutions land in `solutions/` in Phase 2.)

1. **A second region.** Recreate the bucket in `eu-west-1` instead of `us-east-1`. 🔮 Predict what happens if you *omit* the `CreateBucketConfiguration` — then add it and confirm. Why does `us-east-1` behave differently?
2. **Query, not scan.** Add a second session item, then read *only* `sess-001` back by key. 🔮 Predict what it would cost (conceptually) to instead fetch “all sessions for user `ada`” with this key schema — and what a Global Secondary Index would change.
3. **Visibility timeout.** Send a task, `receive` it but **don't** delete it. 🔮 Predict whether a second `receive` (after the visibility window) sees it again. Tie your answer back to the at-least-once pitfall.
4. **Dead-letter intuition.** In two sentences, describe what a *redrive policy / dead-letter queue* would do for a task that fails every time, and which Ch 31 idea it maps to.

In [ ]:
# Exercise 1 — create the bucket in eu-west-1; predict the LocationConstraint behavior.

In [ ]:
# Exercise 2 — put a second session; read sess-001 by key; reason about a GSI.

In [ ]:
# Exercise 3 — receive-without-delete; observe redelivery after the visibility window.

## Next

- ▶️ **Next notebook:** [`33-03-bedrock-call-and-capstone-deploy-notes.ipynb`](./33-03-bedrock-call-and-capstone-deploy-notes.ipynb) — a **mocked Bedrock** model call in the Anthropic Claude shape, then the full component→service **deploy map** for the capstone.
- 🔧 **The production version:** the data and queue plane you drove here is stood up for real by the Ch 36 Terraform ([`templates/terraform-module/`](../../templates/terraform-module/)) and consumed by [`capstone-project/infra/`](../../capstone-project/) — same S3/DynamoDB/SQS, declared as code.
- 📖 **Book:** §33.4 (storage and data) and §33.5 (messaging), and the §33.9 deploy map that the next notebook renders in full.